This notebook contain the code of calculating the inflation of 2025 using the index values of 2025 and the index values of 2024 and taking the median where the combination of the data is not available.  

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
from pathlib import Path

In [3]:
data = pd.read_csv(R"C:\Users\himan\Education1\Projects\MoSPI_CPI\Data\merged\merged_cpi_rebased_2012.csv")
data.head()


,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2012,2014,January,All India,Combined,Food and Beverages,Cereals and Products,119,10.33
1,2012,2014,January,All India,Combined,Food and Beverages,Meat and Fish,118,10.72
2,2012,2014,January,All India,Combined,Food and Beverages,Egg,124,12.82
3,2012,2014,January,All India,Combined,Food and Beverages,Fruits,113,10.37
4,2012,2014,January,All India,Combined,Food and Beverages,Vegetables,122,19.57


In [4]:
# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

data.columns = data.columns.str.strip()

print("Columns:")
print(data.columns.tolist())

Columns:
['BYear', 'Year_i', 'Month_i', 'State_i', 'Sector', 'Group_i', 'SubGroup', 'Index_i', 'Inflation']


In [5]:
# ============================================================
# 4. MAKE SURE YEAR AND INDEX ARE NUMERIC
# ============================================================

data["Year_i"] = pd.to_numeric(
    data["Year_i"],
    errors="coerce"
)

data["Index_i"] = pd.to_numeric(
    data["Index_i"],
    errors="coerce"
)

In [6]:
# ============================================================
# 5. CLEAN THE MATCHING COLUMNS
# ============================================================

keys = [
    "Month_i",
    "State_i",
    "Sector",
    "Group_i",
    "SubGroup"
]

for col in keys:
    data[col] = (
        data[col]
        .astype(str)
        .str.strip()
    )


# ============================================================
# 6. CHECK 2024 AND 2025 ROW COUNTS
# ============================================================

print("\nNumber of rows:")

print(
    data["Year_i"]
    .value_counts()
    .loc[[2024, 2025]]
)


Number of rows:
Year_i
2024    20376
2025    25080
Name: count, dtype: int64


In [7]:

# ============================================================
# 7. SEPARATE 2024 AND 2025 DATA
# ============================================================

data_2024 = data[
    data["Year_i"] == 2024
].copy()

data_2025 = data[
    data["Year_i"] == 2025
].copy()


print("\n2024 rows:", len(data_2024))
print("2025 rows:", len(data_2025))


2024 rows: 20376
2025 rows: 25080


In [8]:
# ============================================================
# 8. FIND MEDIAN OF 2024 INDEX
# ============================================================

median_2024 = data_2024["Index_i"].median()

if pd.isna(median_2024):
    raise ValueError(
        "No valid 2024 Index_i values found."
    )

print("\nMedian of 2024 Index_i:", median_2024)


Median of 2024 Index_i: 190.0


In [9]:
# ============================================================
# 9. CREATE 2024 LOOKUP TABLE
# ============================================================

lookup_2024 = (
    data_2024
    .set_index(keys)["Index_i"]
)


# ============================================================
# 10. FIND 2024 INDEX FOR EVERY 2025 ROW
# ============================================================

# Create the same multi-index for 2025
index_2025_keys = pd.MultiIndex.from_frame(
    data_2025[keys]
)

# Find corresponding 2024 Index_i
previous_index = lookup_2024.reindex(
    index_2025_keys
)

In [10]:
# ============================================================
# 11. CHECK WHICH 2025 ROWS HAVE NO 2024 MATCH
# ============================================================

missing_match = previous_index.isna()

print(
    "\n2025 rows without matching 2024 combination:",
    missing_match.sum()
)


2025 rows without matching 2024 combination: 9360


In [11]:
# ============================================================
# 12. REPLACE MISSING 2024 VALUES WITH 2024 MODE
# ============================================================

previous_index = previous_index.fillna(
    median_2024
)

In [12]:
# ============================================================
# 13. CALCULATE 2025 INFLATION
# ============================================================

inflation = (
    (
        data_2025["Index_i"].values
        - previous_index.values
    )
    /
    previous_index.values
) * 100

inflation = inflation.round(2)

In [13]:
# ============================================================
# 15. PUT CALCULATED INFLATION INTO INFLATION COLUMN
# ============================================================

# Make sure Inflation can store decimal values
data["Inflation"] = pd.to_numeric(
    data["Inflation"],
    errors="coerce"
).astype(float)



In [14]:
# ============================================================
# 15. REPLACE 2025 Index_i WITH INFLATION
# ============================================================
data.loc[
    data["Year_i"] == 2025,
    "Inflation"
] = inflation

In [15]:
data[(data["Year_i"] == 2025)].head(15)

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
215961,2012,2025,January,All India,Combined,Clothing and Footwear,Clothing,196,2.62
215962,2012,2025,January,All India,Combined,Clothing and Footwear,Footwear,194,6.59
215963,2012,2025,January,All India,Combined,Food and Beverages,Cereals and Products,197,5.35
215964,2012,2025,January,All India,Combined,Food and Beverages,Egg,194,-5.37
215965,2012,2025,January,All India,Combined,Food and Beverages,Fruits,199,16.37
215966,2012,2025,January,All India,Combined,Food and Beverages,Meat and Fish,194,-8.92
215967,2012,2025,January,All India,Combined,Food and Beverages,Non-alcoholic Beverages,194,9.60
215968,2012,2025,January,All India,Combined,Food and Beverages,"Prepared Meals, Snacks, Sweets etc.",190,-4.04
215969,2012,2025,January,All India,Combined,Food and Beverages,Residual,203,6.84
215970,2012,2025,January,All India,Combined,Food and Beverages,Vegetables,182,-6.67


In [16]:
print(
    "\nMissing Inflation values:",
    data["Inflation"].isna().sum()
)


Missing Inflation values: 0


In [17]:
data.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2012,2014,January,All India,Combined,Food and Beverages,Cereals and Products,119,10.33
1,2012,2014,January,All India,Combined,Food and Beverages,Meat and Fish,118,10.72
2,2012,2014,January,All India,Combined,Food and Beverages,Egg,124,12.82
3,2012,2014,January,All India,Combined,Food and Beverages,Fruits,113,10.37
4,2012,2014,January,All India,Combined,Food and Beverages,Vegetables,122,19.57


In [18]:
data.shape

(255671, 9)

In [19]:
data.isnull().sum()

BYear        0
Year_i       0
Month_i      0
State_i      0
Sector       0
Group_i      0
SubGroup     0
Index_i      0
Inflation    0
dtype: int64

In [20]:
data.to_csv(r"C:\Users\himan\Education1\Projects\MoSPI_CPI\Data\merged\merged_cpi_cal_index_25.csv", index=False)